# 🚧 YOLOv5 Fine-Tuning — Construction Equipment Detection

This notebook fine-tunes YOLOv5s on a custom construction equipment dataset from Roboflow.

**Before running:**
1. Enable GPU: **Runtime → Change runtime type → T4 GPU → Save**
2. Add your Roboflow API key to Colab Secrets (see Step 2)
3. Then **Runtime → Run all**


## Step 1: Check GPU is available

In [ ]:
import torch

print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')

if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')
    print('✅ Ready to train!')
else:
    print('❌ ERROR: GPU not detected! Go to Runtime > Change runtime type > T4 GPU')

## Step 2: Add your Roboflow API key securely

**Do NOT paste your API key directly into this notebook.**

Instead:
1. Click the 🔑 **Secrets** icon in the left sidebar (looks like a key)
2. Click **+ Add new secret**
3. Name: `ROBOFLOW_API_KEY`
4. Value: paste your actual Roboflow API key
5. Toggle **Notebook access** to ON
6. Then run this cell

In [ ]:
from google.colab import userdata

ROBOFLOW_API_KEY = userdata.get('ROBOFLOW_API_KEY')

if ROBOFLOW_API_KEY:
    print('✅ API key loaded successfully!')
else:
    print('❌ API key not found. Follow the instructions in Step 2.')

## Step 3: Clone YOLOv5 and install dependencies

In [ ]:
import os

# Clone YOLOv5 repository
if not os.path.exists('yolov5'):
    !git clone https://github.com/ultralytics/yolov5

%cd yolov5
!pip install -q -r requirements.txt
!pip install -q roboflow

print('✅ YOLOv5 and dependencies installed!')

## Step 4: Download dataset from Roboflow

In [ ]:
from roboflow import Roboflow

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace("aaraans-workspace").project("construction-equipment-6r96y-iraoo")
version = project.version(1)
dataset = version.download("yolov5")

print(f'✅ Dataset downloaded to: {dataset.location}')
print(f'Dataset name: {dataset.name}')

## Step 5: Verify dataset structure

In [ ]:
import os

dataset_path = dataset.location
print(f'Dataset location: {dataset_path}')
print(f'\nContents:')

for split in ['train', 'valid', 'test']:
    split_path = os.path.join(dataset_path, split, 'images')
    if os.path.exists(split_path):
        count = len(os.listdir(split_path))
        print(f'  {split}: {count} images')

# Show data.yaml contents
yaml_path = os.path.join(dataset_path, 'data.yaml')
print(f'\ndata.yaml contents:')
with open(yaml_path, 'r') as f:
    print(f.read())

## Step 6: Fine-tune YOLOv5s on construction equipment dataset

This will train for 50 epochs on the T4 GPU — expected time: ~30-45 minutes.

In [ ]:
import os

yaml_path = os.path.join(dataset.location, 'data.yaml')

!python train.py \
    --img 640 \
    --batch 16 \
    --epochs 50 \
    --data {yaml_path} \
    --weights yolov5s.pt \
    --name construction_equipment_yolov5s \
    --cache

print('✅ Training complete!')

## Step 7: Evaluate model and extract metrics

In [ ]:
# Run validation on the test set
!python val.py \
    --weights runs/train/construction_equipment_yolov5s/weights/best.pt \
    --data {yaml_path} \
    --img 640 \
    --task test

print('✅ Evaluation complete!')

## Step 8: Display final results — SAVE THESE NUMBERS!

In [ ]:
import pandas as pd
import os

results_path = 'runs/train/construction_equipment_yolov5s/results.csv'

if os.path.exists(results_path):
    results = pd.read_csv(results_path)
    results.columns = results.columns.str.strip()

    # Get best epoch metrics
    best_map50 = results['metrics/mAP_0.5'].max()
    best_map50_95 = results['metrics/mAP_0.5:0.95'].max()
    best_precision = results['metrics/precision'].max()
    best_recall = results['metrics/recall'].max()
    best_epoch = results['metrics/mAP_0.5'].idxmax() + 1

    print('=' * 60)
    print('FINAL TRAINING RESULTS — SAVE THESE NUMBERS!')
    print('=' * 60)
    print(f'  Best Epoch:        {best_epoch}/50')
    print(f'  mAP@0.5:           {best_map50:.4f} ({best_map50*100:.1f}%)')
    print(f'  mAP@0.5:0.95:      {best_map50_95:.4f} ({best_map50_95*100:.1f}%)')
    print(f'  Precision:         {best_precision:.4f} ({best_precision*100:.1f}%)')
    print(f'  Recall:            {best_recall:.4f} ({best_recall*100:.1f}%)')
    print('=' * 60)
    print('\n✅ Copy these numbers into your README and resume!')
else:
    print('Results file not found. Check that training completed successfully.')

## Step 9: Plot training curves

In [ ]:
from IPython.display import Image as IPImage
import os

results_img = 'runs/train/construction_equipment_yolov5s/results.png'

if os.path.exists(results_img):
    display(IPImage(results_img, width=900))
else:
    print('Results plot not found.')

## Step 10: Save results and download model weights

In [ ]:
import json

# Save metrics to JSON
metrics = {
    'model': 'YOLOv5s',
    'dataset': 'Construction Equipment Detection',
    'num_images': 3648,
    'num_classes': 19,
    'epochs': 50,
    'image_size': 640,
    'best_epoch': int(best_epoch),
    'mAP_50': round(float(best_map50), 4),
    'mAP_50_95': round(float(best_map50_95), 4),
    'precision': round(float(best_precision), 4),
    'recall': round(float(best_recall), 4)
}

with open('training_results.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print('Results saved to training_results.json')
print(json.dumps(metrics, indent=2))
print('\n📥 Download these files from the Files panel (left sidebar):')
print('  - training_results.json (metrics)')
print('  - runs/train/construction_equipment_yolov5s/weights/best.pt (model weights)')
print('  - runs/train/construction_equipment_yolov5s/results.png (training curves)')